# Global University
## Ciberseguridad y Desarrollo de Software

<br>

**Materia:** Inteligencia Artificial y Machine Learning

**Equipo:** Hector Oropeza Pelcastre, Sharon Daniela Escobedo Davila, Diego David Lara Martinez

**Profesor:** Jorge Antonio Delgado Magallanes

---

# 03 - Transfer Learning, Applicable Use Case and Demo
## Automatic Fruit Classification using Computer Vision

**Dataset:** [Fruits Classification](https://www.kaggle.com/datasets/utkarshsaxenadn/fruits-classification) - Kaggle

**Classes:** Apple - Banana - Grape - Mango - Strawberry

---
## Notebook Objective

This notebook covers the third stage of the project (**Parcial 3**): improving accuracy via transfer learning with a pretrained ResNet18, building an applicable use case (an interactive Gradio demo), and comparing all three models trained so far.

The main objectives of this notebook are:

1. Clone the project repository and install dependencies (now including Gradio).
2. Download the real dataset and reproduce the same resize/split pipeline used in Parcial 2.
3. Recap the balancing justification from Parcial 2 (no changes needed here).
4. Fine-tune a pretrained ResNet18 (frozen backbone + new classifier head) and report metrics.
5. Compare the base CNN, the Optuna-tuned CNN (both from Parcial 2), and this ResNet18 model.
6. Launch a Gradio demo as the applicable use case: upload a fruit photo, get the predicted class and confidence.

**Note:** This notebook is designed to run in Google Colab with a GPU runtime (Runtime > Change runtime type > GPU). Google Drive mounting is optional and known to be unreliable in some environments — the notebook works fully without it (see Section 3).

## 1. Clone Repository and Install Dependencies

In [ ]:
!git clone https://github.com/GoldenDiegos/fruits-classifier.git
%cd fruits-classifier/fruit_neural_network_project/project
!pip install -r requirements.txt -q

import sys
from pathlib import Path

# Make sure local packages (models/, training/, evaluation/, deployment/, ...) are importable
sys.path.insert(0, str(Path.cwd()))
print("Working directory:", Path.cwd())

## 2. Library Imports

In [ ]:
import os
import shutil
import random
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage, Markdown

from PIL import Image
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

## 3. Google Drive Mount (Dataset Cache, Optional)

Same optional pattern as `02_Training_Colab.ipynb`: if Drive mounting fails (as it has consistently for this environment), the notebook falls back to no caching and a browser-download export at the end. Uses a distinct cache folder from Parcial 2's.

In [ ]:
DRIVE_AVAILABLE = False
DRIVE_CACHE_DIR = None
DRIVE_SPLIT_ZIP = None

try:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_CACHE_DIR = Path("/content/drive/MyDrive/fruits_classifier_parcial3")
    DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_SPLIT_ZIP = DRIVE_CACHE_DIR / "data_split.zip"
    DRIVE_AVAILABLE = True

    print(f"Drive cache path: {DRIVE_SPLIT_ZIP}")
    print(f"Cache exists: {DRIVE_SPLIT_ZIP.exists()}")
except Exception as e:
    print(f"Google Drive mount failed ({e}). Continuing without a cache.")
    print("You'll download/resize/split the dataset once in this session and keep going without restarting.")

## 4. Global Configuration

Same configuration values used in Parcial 1 and Parcial 2, pointed at this project's own `data/` folder.

In [ ]:
# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Image preprocessing
IMAGE_SIZE = (224, 224)

# Dataset split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

assert round(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 2) == 1.00, "Split ratios must sum to 1.0"

IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png"]

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
SPLIT_DIR = Path("data/split")

print("Configuration loaded successfully.")
print(f"Image size          : {IMAGE_SIZE}")
print(f"Split ratios        : train={TRAIN_RATIO} | val={VAL_RATIO} | test={TEST_RATIO}")
print(f"Random seed         : {RANDOM_SEED}")
print(f"Raw directory       : {RAW_DIR}")
print(f"Processed directory : {PROCESSED_DIR}")
print(f"Split directory     : {SPLIT_DIR}")

## 5. Restore Cached Split (Optional)

If a cached split already exists on Drive from a previous session, restore it and skip straight to Section 10. Otherwise (or if Drive isn't available), continue with the Kaggle download below.

In [ ]:
if DRIVE_AVAILABLE and DRIVE_SPLIT_ZIP.exists():
    print("Cached split found on Drive. Restoring data/split ...")
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(DRIVE_SPLIT_ZIP), str(SPLIT_DIR))
    print("Restored. You can skip ahead to Section 10 if this looks correct.")
elif DRIVE_AVAILABLE:
    print("No cached split found on Drive yet. Continue with the download/resize/split sections below.")
else:
    print("Drive not available, no cache to restore. Continue with the download/resize/split sections below.")

## 6. Kaggle Setup and Dataset Download

Same process as Parcial 1 and Parcial 2: upload your `kaggle.json` API token when prompted.

In [ ]:
!pip install kaggle -q

from google.colab import files

print("Upload your kaggle.json file:")
uploaded = files.upload()

if "kaggle.json" not in uploaded:
    raise FileNotFoundError("kaggle.json was not uploaded. Please upload your Kaggle API token file.")

os.makedirs("/root/.config/kaggle", exist_ok=True)
!cp kaggle.json /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json

print("Kaggle credentials configured successfully.")

if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)

for folder in [RAW_DIR, PROCESSED_DIR, SPLIT_DIR / "train", SPLIT_DIR / "val", SPLIT_DIR / "test"]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "utkarshsaxenadn/fruits-classification"

!kaggle datasets download -d {DATASET_SLUG} -p {str(RAW_DIR)} --unzip

print("Dataset downloaded and extracted successfully.")

## 7. Class Folder Detection

Same detection logic as Parcial 1 and 2 — class folder names are confirmed singular (Apple, Banana, Grape, Mango, Strawberry).

In [ ]:
candidate_class_dirs = []

for folder in RAW_DIR.rglob("*"):
    if folder.is_dir():
        image_files = [f for f in folder.glob("*") if f.suffix.lower() in IMAGE_EXTENSIONS]
        if len(image_files) > 0:
            candidate_class_dirs.append(folder)

if not candidate_class_dirs:
    raise FileNotFoundError("No image folders were found inside RAW_DIR.")

CLASS_DIRS = sorted(candidate_class_dirs)
UNIQUE_CLASS_NAMES = sorted({class_dir.name for class_dir in CLASS_DIRS})

print("Image folders selected for processing:")
for class_dir in CLASS_DIRS:
    print(f"  - {class_dir.name}")

print(f"\nTotal unique classes detected: {len(UNIQUE_CLASS_NAMES)}")

## 8. Resize to 224x224 and Split (70/15/15)

Same LANCZOS resize and per-class stratified split as Parcial 1 and 2.

In [ ]:
# Resize
if PROCESSED_DIR.exists():
    shutil.rmtree(PROCESSED_DIR)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

resize_errors = []
total_processed = 0

for class_dir in CLASS_DIRS:
    output_class_dir = PROCESSED_DIR / class_dir.name
    output_class_dir.mkdir(parents=True, exist_ok=True)

    image_files = [f for f in class_dir.glob("*") if f.suffix.lower() in IMAGE_EXTENSIONS]

    for img_path in image_files:
        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
                img = img.resize(IMAGE_SIZE, Image.LANCZOS)
                img.save(output_class_dir / img_path.name)
            total_processed += 1
        except Exception as e:
            resize_errors.append({"file": str(img_path), "class": class_dir.name, "error": str(e)})

print(f"Images processed successfully : {total_processed}")
print(f"Errors                        : {len(resize_errors)}")

# Split
processed_class_dirs = [d for d in sorted(PROCESSED_DIR.iterdir()) if d.is_dir()]

if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
for subset in ["train", "val", "test"]:
    (SPLIT_DIR / subset).mkdir(parents=True, exist_ok=True)

for class_dir in processed_class_dirs:
    images = [f for f in class_dir.glob("*") if f.suffix.lower() in IMAGE_EXTENSIONS]

    train_imgs, temp_imgs = train_test_split(
        images, test_size=(1 - TRAIN_RATIO), random_state=RANDOM_SEED, shuffle=True
    )
    val_imgs, test_imgs = train_test_split(
        temp_imgs, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=RANDOM_SEED, shuffle=True
    )

    for subset_name, subset_imgs in {"train": train_imgs, "val": val_imgs, "test": test_imgs}.items():
        destination_dir = SPLIT_DIR / subset_name / class_dir.name
        destination_dir.mkdir(parents=True, exist_ok=True)
        for img_path in subset_imgs:
            shutil.copy2(img_path, destination_dir / img_path.name)

print("Split completed: data/split/{train,val,test}/<ClassName>/")

## 9. Cache the Split to Google Drive (Optional)

In [ ]:
if DRIVE_AVAILABLE and not DRIVE_SPLIT_ZIP.exists():
    shutil.make_archive(str(DRIVE_SPLIT_ZIP.with_suffix("")), "zip", str(SPLIT_DIR))
    print(f"Cached split saved to: {DRIVE_SPLIT_ZIP}")
elif DRIVE_AVAILABLE:
    print("Cache already exists on Drive, skipping.")
else:
    print("Drive not available, skipping cache step.")

## 10. Balancing Recap

Class balance was already analyzed and justified in Parcial 2 (see `02_Training_Colab.ipynb`, Section 10, and `reports/parcial_2/`): the dataset is perfectly balanced (2,000 images/class), so inverse-frequency class weights come out ~1.0 for every class. `main.py` applies this same weighting automatically for every model trained below (`--use-class-weights` defaults to on) — no changes needed for Parcial 3.

## 11. Transfer Learning: Fine-Tune ResNet18 (Accuracy Improvement)

Unlike Parcial 2 (from-scratch CNN only), pretrained models are allowed in this stage. Uses a ResNet18 pretrained on ImageNet with its backbone frozen and only a new classifier head (`Linear(512, 5)`) trained — this converges fast and is not very hyperparameter-sensitive, so a single direct run (not an Optuna search) is enough.

In [ ]:
!python main.py \
    --train-dir data/split/train \
    --val-dir data/split/val \
    --test-dir data/split/test \
    --model-name resnet18 \
    --epochs 15 \
    --batch-size 32 \
    --learning-rate 1e-3 \
    --output models/checkpoints/resnet18_model.pt \
    --metrics-output reports/parcial_3/resnet18_metrics.json \
    --curves-output reports/parcial_3/resnet18_curves.png \
    --confusion-matrix-output reports/parcial_3/resnet18_confusion_matrix.png

### ResNet18 Results

In [ ]:
with open("reports/parcial_3/resnet18_metrics.json") as f:
    resnet18_metrics = json.load(f)

print(json.dumps(resnet18_metrics, indent=2))
display(IPImage(filename="reports/parcial_3/resnet18_curves.png"))
display(IPImage(filename="reports/parcial_3/resnet18_confusion_matrix.png"))

## 12. Three-Way Comparison: Base CNN vs. Tuned CNN vs. ResNet18

Reads the metrics already committed from Parcial 2 (`reports/parcial_2/`) alongside this notebook's new ResNet18 results, and builds a single comparison table.

In [ ]:
with open("reports/parcial_2/base_model_metrics.json") as f:
    base_cnn_metrics = json.load(f)

with open("reports/parcial_2/tuned_model_metrics.json") as f:
    tuned_cnn_metrics = json.load(f)

rows = [
    ("Base CNN (from scratch)", base_cnn_metrics["test_metrics"]),
    ("Tuned CNN (Optuna)", tuned_cnn_metrics["test_metrics"]),
    ("ResNet18 (transfer learning)", resnet18_metrics["test_metrics"]),
]

lines = [
    "# Parcial 3 - Three-Way Model Comparison",
    "",
    "| Model | Accuracy | Precision (macro) | Recall (macro) | F1 (macro) |",
    "|---|---:|---:|---:|---:|",
]
for name, metrics in rows:
    lines.append(
        f"| {name} | {metrics['accuracy']:.4f} | {metrics['precision_macro']:.4f} | "
        f"{metrics['recall_macro']:.4f} | {metrics['f1_macro']:.4f} |"
    )

comparison_markdown = "\n".join(lines)

Path("reports/parcial_3").mkdir(parents=True, exist_ok=True)
Path("reports/parcial_3/comparison_table.md").write_text(comparison_markdown, encoding="utf-8")

display(Markdown(comparison_markdown))

## 13. Applicable Use Case: Gradio Demo

A minimal interactive demo: upload a fruit photo, get back the predicted class and a probability breakdown across all 5 classes. Runs `scripts/gradio_demo.py`, reusing `FruitPredictor` (which reconstructs the exact architecture from the checkpoint's own metadata). `--share` creates a temporary public link — useful for a live demo or a short screen recording for the presentation.

Run this cell, click the public URL it prints, upload a photo, and try it out. Interrupt the cell (or just move on) when done.

In [ ]:
!python scripts/gradio_demo.py --model-path models/checkpoints/resnet18_model.pt --share

## 14. Export Final Model

Same no-Drive-dependency export pattern as Parcial 2: copies to Drive if available, otherwise downloads straight to the browser.

In [ ]:
resnet18_checkpoint_path = Path("models/checkpoints/resnet18_model.pt")

if DRIVE_AVAILABLE:
    drive_model_path = DRIVE_CACHE_DIR / "resnet18_model.pt"
    shutil.copy(resnet18_checkpoint_path, drive_model_path)
    print(f"Copied final model to: {drive_model_path}")
else:
    from google.colab import files
    print("Drive not available - triggering a browser download instead.")
    files.download(str(resnet18_checkpoint_path))

import torch
checkpoint = torch.load(resnet18_checkpoint_path, map_location="cpu")
print("Checkpoint metadata:")
print(json.dumps(checkpoint["metadata"], indent=2, default=str))

## 15. Conclusions

- Pretrained ResNet18 (frozen backbone + new classifier head) was fine-tuned on the same real 10,000-image split used throughout the project, using the same class-weighted loss established in Parcial 2.
- Results are compared directly against both Parcial 2 models (base CNN and Optuna-tuned CNN) in `reports/parcial_3/comparison_table.md`.
- The applicable use case — an interactive Gradio demo — lets anyone upload a fruit photo and get a live prediction with confidence, without needing any API client.
- The final model was exported as `models/checkpoints/resnet18_model.pt`, including metadata with all training hyperparameters and final test metrics.